# Machote — Módulo 3: Machine Learning
### Base reutilizable para todos los ejercicios del módulo

---
## ¿Cómo usar este archivo?

1. **Corre las celdas en orden** de arriba hacia abajo (Celda 1, 2, 3...)
2. Cada sección tiene su propio bloque de código con comentarios
3. Para un ejercicio nuevo: copia solo las celdas que necesites

## El flujo de ML — siempre el mismo

```
 1. Cargar datos          → obtener X (características) e Y (etiquetas)
 2. Explorar (EDA)        → entender qué hay antes de modelar
 3. Seleccionar features  → quedarse con las columnas útiles
 4. Dividir               → train / test (nunca mezclarlos)
 5. Escalar               → poner todos los valores en la misma escala
 6. Crear modelo          → ← aquí cambia según el tema de la clase
 7. Entrenar              → modelo.fit(X_train, y_train)
 8. Predecir              → Y_pred = modelo.predict(X_test)
 9. Evaluar               → medir accuracy, ver errores
10. Comparar              → comparar varios modelos entre sí
```

Los pasos 1-5 y 7-10 son casi idénticos siempre.
**Solo cambia el paso 6** según qué tema estés viendo.

In [1]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports y path
# Corre esta celda siempre primero
# ═══════════════════════════════════════════════════════════════
import sys, os

# Ruta relativa al machote — funciona en Windows y Linux sin cambiar nada
# (sube dos niveles desde donde está este notebook)
RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

# Si lo anterior da error, descomenta la línea de tu sistema:
# sys.path.append(r"D:/Proyectos/Diplomado-RNA/Machote")            # Windows
# sys.path.append("/home/usr-lbr-maq20/Documentos/Diplomado-RNA/Machote")  # Linux

# ── Imports de uso general ────────────────────────────────────
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Intentar importar el machote (opcional) ───────────────────
try:
    from machote_ML import *
    print("Machote importado ✅")
except ImportError:
    print("Machote no encontrado — usando funciones locales de este notebook")

print("Imports listos")

✅ machote_ML cargado correctamente

Funciones disponibles:
  CARGAR:   cargar_desde_sklearn | cargar_desde_uci | cargar_desde_csv
  EXPLORAR: resumen_rapido | ver_distribucion | ver_correlacion_con_y | ver_mapa_calor
  PREPARAR: seleccionar_features | dividir_datos | escalar_datos | binarizar_datos
  EVALUAR:  evaluar_clasificacion | evaluar_regresion | comparar_modelos
Machote importado ✅
Imports listos


---
# BLOQUE 1 — Cargar datos

Hay tres formas comunes. Elige la que aplique a tu ejercicio.
Siempre al final tendrás: **X** (DataFrame con características) e **Y** (array con etiquetas).

In [ ]:
# ── OPCIÓN A: Dataset de sklearn (para practicar, sin internet) ────────
# Datasets disponibles para clasificación:
#   load_breast_cancer → maligno/benigno (el que ya conocemos)
#   load_iris          → 3 tipos de flor
#   load_wine          → tipos de vino
# Datasets para regresión (predecir un número):
#   load_diabetes      → nivel de diabetes
#   fetch_california_housing → precio de casas

from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
Y = dataset.target  # 0 = maligno, 1 = benigno
nombres_clases = list(dataset.target_names)

print(f"Dataset: {type(dataset).__name__}")
print(f"Forma de X: {X.shape}  ({X.shape[0]} muestras, {X.shape[1]} características)")
print(f"Clases: {nombres_clases}")
print(f"Distribución Y: {dict(zip(*np.unique(Y, return_counts=True)))}")

In [ ]:
# ── OPCIÓN B: Dataset de UCI (requiere internet) ───────────────────────
# Descomenta si vas a usar UCI en lugar de sklearn

# from ucimlrepo import fetch_ucirepo
# dataset = fetch_ucirepo(id=17)                         # 17 = Breast Cancer Wisconsin
# X = dataset.data.features                             # ya viene como DataFrame
# Y_raw = dataset.data.targets
# print("Columnas en Y:", Y_raw.columns.tolist())        # ver nombre exacto de columna
# Y = Y_raw['Diagnosis'].map({'M': 0, 'B': 1}).to_numpy()
# nombres_clases = ['Maligno', 'Benigno']

# ── OPCIÓN C: CSV propio ────────────────────────────────────────────────
# Descomenta si tienes un archivo CSV

# df = pd.read_csv(r"ruta/a/tu/archivo.csv")
# print(df.head())             # ver primeras filas
# print(df.columns.tolist())   # ver nombres de columnas
# X = df.drop(columns=['nombre_columna_Y'])
# Y = df['nombre_columna_Y'].values

print("(Celdas de UCI y CSV comentadas — descomenta la que necesites)")

---
# BLOQUE 2 — Explorar datos (EDA)

**Siempre explorar antes de modelar.** Muchos errores se evitan mirando los datos primero.
El EDA responde: ¿cuántos datos tengo? ¿hay valores faltantes? ¿qué columnas importan?

In [ ]:
# ── 2a. Resumen rápido ────────────────────────────────────────────────
print("="*55)
print("  RESUMEN DEL DATASET")
print("="*55)
print(f"  Muestras:              {X.shape[0]}")
print(f"  Características:       {X.shape[1]}")

# Valores faltantes (NaN) — si hay, hay que manejarlos antes de entrenar
nan_total = X.isnull().sum().sum()
if nan_total == 0:
    print(f"  Valores faltantes:     Ninguno ✅")
else:
    print(f"  Valores faltantes:     {nan_total} ⚠️")
    print(X.isnull().sum()[X.isnull().sum() > 0])

# Distribución de clases — desbalance puede engañar el accuracy
print(f"\n  Distribución de Y:")
vals, cnts = np.unique(Y, return_counts=True)
for v, c in zip(vals, cnts):
    barra = '█' * int(c / len(Y) * 25)
    alerta = " ⚠️ DESBALANCE" if c / len(Y) > 0.8 else ""
    nombre = nombres_clases[v] if 'nombres_clases' in dir() and v < len(nombres_clases) else str(v)
    print(f"    {nombre} ({v}): {c} ({c/len(Y)*100:.1f}%) {barra}{alerta}")
print("="*55)

In [ ]:
# ── 2b. Ver las primeras filas como tabla ─────────────────────────────
# .head() muestra las primeras 5 filas como tabla HTML en VSCode/Jupyter
X.head()

In [ ]:
# ── 2c. Estadísticas básicas ──────────────────────────────────────────
# count=cuántos, mean=promedio, std=qué tan dispersos, min/max=rango
# Sirve para entender los rangos ANTES de escalar o binarizar
X.describe()

In [ ]:
# ── 2d. Correlación con Y — ¿qué columnas importan? ──────────────────
# Una correlación alta (|valor| > 0.5) indica que esa columna ayuda a predecir Y
# Positiva (+): cuando sube la característica, sube Y
# Negativa (-): cuando sube la característica, baja Y
# Cerca de 0: esa columna no aporta — considerar eliminarla

df_temp = X.copy()
df_temp['__Y__'] = Y
correlaciones = df_temp.corr()['__Y__'].drop('__Y__').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, max(4, len(correlaciones) * 0.3)))
colors = ['#D85A30' if abs(v) >= 0.5 else '#aaaaaa' for v in correlaciones]
correlaciones.plot(kind='barh', ax=ax, color=colors, edgecolor='none')
ax.axvline(x=0,    color='gray',  linewidth=0.8)
ax.axvline(x=0.5,  color='green', linewidth=1.2, linestyle='--', label='umbral ±0.5')
ax.axvline(x=-0.5, color='green', linewidth=1.2, linestyle='--')
ax.set_title('Correlación de cada característica con Y\n(naranja = útiles, gris = posible ruido)')
ax.legend()
plt.tight_layout()
plt.show()

# Dirección dominante — importante para saber si los pesos serán + o -
prom = correlaciones.mean()
print(f"\nCorrelación promedio: {prom:+.4f}")
if prom < 0:
    print("→ Mayoría NEGATIVA: valores altos en X tienden a Y bajo")
else:
    print("→ Mayoría POSITIVA: valores altos en X tienden a Y alto")

In [ ]:
# ── 2e. Mapa de calor — detectar columnas redundantes ─────────────────
# Si dos columnas tienen correlación 0.95+, dicen casi lo mismo → puedes quitar una
# La columna 'diagnostico_Y' es Y — ver qué tan correlacionadas están con ella

top15 = correlaciones.abs().head(15).index.tolist()  # las 15 más útiles
df_calor = X[top15].copy()
df_calor['diagnostico_Y'] = Y

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    df_calor.corr(),
    annot=True, fmt='.2f',
    cmap='RdBu_r',        # rojo=alta correlación, azul=correlación negativa
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.3,
    ax=ax
)
ax.set_title('Mapa de calor — top 15 características + Y')
plt.tight_layout()
plt.show()

---
# BLOQUE 3 — Preparar datos

**Regla de oro:** preparar train y test **por separado**.
Si los preparas juntos, el test "sabe" cosas del train → el modelo hace trampa → accuracy falso.

In [ ]:
# ── 3a. Seleccionar características útiles ────────────────────────────
# Nos quedamos solo con columnas que tengan |correlación| >= umbral
# El umbral es una decisión tuya: empieza con 0.5, ajusta si necesitas

UMBRAL = 0.5  # ← ajusta aquí si quedan demasiadas o muy pocas

features_utiles = correlaciones[correlaciones.abs() >= UMBRAL].index.tolist()
X_reducido = X[features_utiles]

print(f"Selección con umbral={UMBRAL}:")
print(f"  Conservadas: {len(features_utiles)} de {X.shape[1]}")
print(f"  Eliminadas:  {X.shape[1] - len(features_utiles)}")
print(f"  Columnas:    {features_utiles}")

In [ ]:
# ── 3b. Dividir en train y test ───────────────────────────────────────
from sklearn.model_selection import train_test_split

# test_size=0.25  → 25% para test, 75% para train
# stratify=Y      → mantiene misma proporción de clases en train y test
# random_state=42 → fija la aleatoriedad (reproducible, cualquier número sirve)
X_train, X_test, y_train, y_test = train_test_split(
    X_reducido, Y,
    test_size=0.25,
    stratify=Y,
    random_state=42
)

print(f"Train: {len(X_train)} muestras  |  Test: {len(X_test)} muestras")
print(f"Distribución train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Distribución test:  {dict(zip(*np.unique(y_test,  return_counts=True)))}")

In [ ]:
# ── 3c. Escalar los datos ─────────────────────────────────────────────
# Escalar = poner todos los valores en la misma escala
# Sin escalar, columnas con números grandes (ej: area=1001) dominan el modelo
# sobre columnas pequeñas (ej: smoothness=0.11) aunque no sean más importantes
#
# StandardScaler: resta la media y divide entre desviación estándar
#   → resultado: valores centrados en 0, mayoría entre -3 y +3
#   → ideal para: regresión logística, SVM, redes neuronales
#
# IMPORTANTE: fit SOLO con train, transform en ambos
#   fit_transform(train) → aprende la escala del train y la aplica
#   transform(test)      → aplica la misma escala aprendida (sin aprender de nuevo)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # aprende + transforma
X_test_sc  = scaler.transform(X_test)       # solo transforma (usa la misma escala)

print(f"Escalado. X_train_sc: {X_train_sc.shape}")
print(f"  Media en train (debe ser ~0):  {X_train_sc.mean():.6f}")
print(f"  Std en train (debe ser ~1):    {X_train_sc.std():.6f}")

---
# BLOQUE 4 — Modelos del Módulo 3

Cada sección es independiente. Usa la que corresponda al tema de la clase.

## Tema 3 — Regresión Lineal

**Cuándo usar:** cuando Y es un número continuo (precio, temperatura, velocidad).

**Fórmula:** `ŷ = w₀ + w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ`

La misma idea que el perceptrón pero **sin función de activación** — la suma ponderada ES la predicción.
El modelo aprende los pesos minimizando el error cuadrático medio (MSE).

In [ ]:
# ── Regresión Lineal ──────────────────────────────────────────────────
# Para demostrar regresión, usamos diabetes (Y es un número continuo)
# Si tu ejercicio tiene Y continuo, usa este bloque directamente

from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error, r2_score

# Cargar dataset de regresión
diab = load_diabetes()
X_reg = pd.DataFrame(diab.data, columns=diab.feature_names)
Y_reg = diab.target  # número continuo (no 0/1)

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, Y_reg, test_size=0.25, random_state=42)
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc = sc.transform(X_te)

# Crear y entrenar el modelo
modelo_lr = LinearRegression()
modelo_lr.fit(X_tr_sc, y_tr)      # aprende los pesos

# Predecir y evaluar
Y_pred_lr = modelo_lr.predict(X_te_sc)

mse  = mean_squared_error(y_te, Y_pred_lr)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, Y_pred_lr)

print("Regresión Lineal — Dataset Diabetes")
print(f"  MSE:  {mse:.2f}   (error cuadrático — penaliza errores grandes)")
print(f"  RMSE: {rmse:.2f}  (raíz del error — mismas unidades que Y)")
print(f"  R²:   {r2:.4f}  (1.0=perfecto, 0.0=no explica nada)")
print()
print(f"  Pesos aprendidos (coef_): {modelo_lr.coef_[:3]}...")
print(f"  Intercepto (bias):        {modelo_lr.intercept_:.4f}")

# Gráfica real vs predicho
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(y_te, Y_pred_lr, alpha=0.5, s=20, color='#3B8BD4')
lims = [min(y_te.min(), Y_pred_lr.min()), max(y_te.max(), Y_pred_lr.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
ax.set_xlabel('Valores reales')
ax.set_ylabel('Valores predichos')
ax.set_title(f'Real vs Predicho — R²: {r2:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

## Tema 3 — Regresión Logística

**Cuándo usar:** cuando Y es una categoría (0/1, spam/no spam, maligno/benigno).

A pesar del nombre, **clasifica**, no regresa un número.
Usa la función **sigmoide** para convertir la suma ponderada en probabilidad.

**Fórmula:** `P(y=1) = sigmoide(w₀ + w₁·x₁ + ... + wₙ·xₙ)`

Si P >= 0.5 → predice clase 1. Si P < 0.5 → predice clase 0.

In [ ]:
# ── Regresión Logística ───────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Crear y entrenar
modelo_log = LogisticRegression(
    max_iter=1000,   # máximo de iteraciones para converger
    random_state=42
)
modelo_log.fit(X_train_sc, y_train)

# Predecir
Y_pred_log = modelo_log.predict(X_test_sc)

# Lo especial de la logística: también da PROBABILIDADES
# probas[:,0] = probabilidad de clase 0 (maligno)
# probas[:,1] = probabilidad de clase 1 (benigno)
probas = modelo_log.predict_proba(X_test_sc)

acc = accuracy_score(y_test, Y_pred_log)
print(f"Regresión Logística — Accuracy: {acc:.4f} ({acc*100:.1f}%)")
print()
print("Reporte completo (precision/recall/F1):")
print(classification_report(y_test, Y_pred_log, target_names=nombres_clases))
print()
print("Primeras 5 predicciones con probabilidades:")
for i in range(5):
    print(f"  Pred={Y_pred_log[i]} Real={y_test[i]} | "
          f"P(maligno)={probas[i][0]:.2f} P(benigno)={probas[i][1]:.2f}")

In [ ]:
# ── Matriz de confusión visual ─────────────────────────────────────────
# TN=predijo 0 y era 0  TP=predijo 1 y era 1  (correctos)
# FP=predijo 1 y era 0  FN=predijo 0 y era 1  (errores)
# En medicina: FP (decir sano cuando está enfermo) es el más peligroso

cm = confusion_matrix(y_test, Y_pred_log)
TN, FP, FN, TP = cm.ravel()

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[f'Pred {n}' for n in nombres_clases],
    yticklabels=[f'Real {n}' for n in nombres_clases],
    ax=ax
)
ax.set_title(f'Matriz de Confusión — {acc*100:.1f}%')
plt.tight_layout()
plt.show()

print(f"  TN (correctamente {nombres_clases[0]}): {TN}")
print(f"  TP (correctamente {nombres_clases[1]}): {TP}")
print(f"  FP (predijo {nombres_clases[1]}, era {nombres_clases[0]}): {FP} ← peligroso en medicina")
print(f"  FN (predijo {nombres_clases[0]}, era {nombres_clases[1]}): {FN}")

## Tema 4 — Árbol de Decisión

**Cuándo usar:** clasificación o regresión cuando necesitas un modelo **interpretable**.

Hace preguntas sucesivas del tipo "¿radio > 15?" → "¿área > 700?" hasta llegar a una decisión.

**Ventaja:** puedes visualizar el árbol y entender exactamente por qué predijo algo.
**Desventaja:** puede memorizar el dataset (overfitting) si el árbol crece demasiado.

In [ ]:
# ── Árbol de Decisión ─────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier, plot_tree

modelo_arbol = DecisionTreeClassifier(
    max_depth=4,      # profundidad máxima del árbol
                      # más profundo = más complejo = más riesgo de overfitting
    random_state=42
)
modelo_arbol.fit(X_train_sc, y_train)  # no necesita escalar, pero lo dejamos igual

Y_pred_arbol = modelo_arbol.predict(X_test_sc)
acc_arbol    = accuracy_score(y_test, Y_pred_arbol)
print(f"Árbol de Decisión — Accuracy: {acc_arbol:.4f} ({acc_arbol*100:.1f}%)")

# Visualizar el árbol
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    modelo_arbol,
    feature_names=features_utiles,  # nombres de las columnas
    class_names=nombres_clases,
    filled=True,                     # colorea los nodos según la clase
    ax=ax
)
plt.title(f'Árbol de Decisión (max_depth={modelo_arbol.max_depth})')
plt.tight_layout()
plt.show()

# Importancia de características — cuáles usó más el árbol
importancias = pd.Series(
    modelo_arbol.feature_importances_,
    index=features_utiles
).sort_values(ascending=False)

print("\nTop 5 características más usadas por el árbol:")
for feat, imp in importancias.head(5).items():
    barra = '█' * int(imp * 40)
    print(f"  {feat:<35} {imp:.4f}  {barra}")

## Tema 4 — SVM (Máquina de Soporte Vectorial)

**Cuándo usar:** datasets pequeños/medianos con muchas características. Muy efectivo.

Encuentra el **hiperplano** (línea/plano de separación) que maximiza el margen entre clases.

Parámetros importantes:
- `kernel`: tipo de frontera de decisión
  - `'linear'` → línea recta (igual que regresión logística)
  - `'rbf'` → curva (más flexible, el más común)
  - `'poly'` → polinomial
- `C`: penalización por errores. Alto C = acepta menos errores = más complejo

In [ ]:
# ── SVM ───────────────────────────────────────────────────────────────
from sklearn.svm import SVC

modelo_svm = SVC(
    kernel='rbf',    # frontera curva (el más común)
    C=1.0,           # penalización por errores
    random_state=42
)
modelo_svm.fit(X_train_sc, y_train)  # ← escalar SIEMPRE antes de SVM

Y_pred_svm = modelo_svm.predict(X_test_sc)
acc_svm    = accuracy_score(y_test, Y_pred_svm)
print(f"SVM (kernel=rbf) — Accuracy: {acc_svm:.4f} ({acc_svm*100:.1f}%)")

## Tema 5 — K-Means (Aprendizaje No Supervisado)

**La diferencia clave:** en aprendizaje supervisado teníamos X e Y.
Aquí **solo tenemos X** — no hay etiquetas. El modelo descubre los grupos por sí solo.

**Cuándo usar:** cuando quieres encontrar grupos naturales en los datos sin saber de antemano cuáles son.

**¿Cuántos clusters (K) usar?** → El **método del codo**: prueba varios K y elige donde la inercia deja de bajar significativamente.

In [ ]:
# ── K-Means ───────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Método del codo — ¿cuántos clusters?
inercias = []
rango_k  = range(2, 9)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_sc)
    inercias.append(km.inertia_)  # inertia = suma de distancias al centroide más cercano

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rango_k, inercias, 'bo-', linewidth=2)
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Inercia')
ax.set_title('Método del codo — elige K donde la curva "dobla"')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("¿Dónde está el 'codo'? Ahí está el K óptimo.")
print("Si la curva no tiene codo claro, los datos no tienen clusters naturales fuertes.")

In [ ]:
# ── K-Means con K elegido ─────────────────────────────────────────────
K = 2  # ← ajusta según lo que viste en el gráfico del codo

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10    # número de inicializaciones aleatorias (elige la mejor)
)
kmeans.fit(X_train_sc)               # ← sin Y, solo aprende de X
clusters = kmeans.predict(X_test_sc) # devuelve 0, 1, 2... (número de cluster)

# Evaluación sin Y real — métricas de cohesión interna
silhouette = silhouette_score(X_test_sc, clusters)
# Silhouette: -1 a +1
# Cerca de +1 = clusters bien separados y compactos
# Cerca de 0  = clusters se solapan
# Cerca de -1 = muestras asignadas al cluster equivocado

print(f"K-Means con K={K}")
print(f"  Silhouette score: {silhouette:.4f}")
print(f"  Muestras por cluster: {dict(zip(*np.unique(clusters, return_counts=True)))}")

# Comparación con Y real (solo para ver si los clusters coinciden con las clases)
print(f"\nComparación clusters vs Y real:")
for cluster in range(K):
    mascara = clusters == cluster
    if mascara.sum() > 0:
        clases_en_cluster = y_test[mascara]
        vals, cnts = np.unique(clases_en_cluster, return_counts=True)
        print(f"  Cluster {cluster}: ", end="")
        for v, c in zip(vals, cnts):
            nombre = nombres_clases[v] if 'nombres_clases' in dir() and v < len(nombres_clases) else str(v)
            print(f"{nombre}={c} ", end="")
        print()

---
# BLOQUE 5 — Comparar todos los modelos

Al final del proyecto, compara los modelos que probaste.

In [ ]:
# ── Comparación visual de modelos ─────────────────────────────────────
# Agrega aquí el accuracy de cada modelo que hayas probado
resultados = {
    'Reg. Logística':  acc,
    'Árbol Decisión': acc_arbol,
    'SVM':            acc_svm,
}

nombres_mod = list(resultados.keys())
valores_mod = list(resultados.values())
mejor_val   = max(valores_mod)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#D85A30' if v == mejor_val else '#3B8BD4' for v in valores_mod]
bars = ax.bar(nombres_mod, [v*100 for v in valores_mod], color=colors, edgecolor='none')

for bar, val in zip(bars, valores_mod):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
ax.set_title('Comparación de modelos')
ax.axhline(y=mejor_val*100, color='orange', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

mejor = max(resultados, key=resultados.get)
print(f"Mejor modelo: {mejor} con {resultados[mejor]*100:.1f}%")

---
# Referencia rápida — Módulo 3

```
TIPO DE PROBLEMA       MODELO A USAR           MÉTRICAS
──────────────────────────────────────────────────────────
Clasificación binaria  LogisticRegression      accuracy, F1, matriz confusión
Clasificación visual   DecisionTreeClassifier  accuracy + plot_tree
Clasificación robusta  SVC                     accuracy
Regresión (número)     LinearRegression        MSE, RMSE, R²
Agrupación sin labels  KMeans                  silhouette_score

PARÁMETROS CLAVE
──────────────────────────────────────────────────────────
test_size=0.25     → 25% test, 75% train
stratify=Y         → misma proporción de clases en train/test
random_state=42    → hace la división reproducible
max_depth=4        → en árbol: controla profundidad (evita overfitting)
C=1.0              → en SVM: penalización por errores
n_clusters=K       → en K-Means: cuántos grupos buscar

ERRORES COMUNES
──────────────────────────────────────────────────────────
No escalar antes de SVM    → resultados muy malos
Escalar test antes de fit  → el test 'sabe' del train (trampa)
No usar stratify           → distribución de clases desigual
K-Means con Y              → K-Means NO recibe Y (no supervisado)
```